# What LLM Evaluation Actually Means
*Vibe testing and why it fails, the systematic/repeatable/criteria-based definition of an LLM eval, why an eval is a full testing setup rather than a single metric, the model-eval vs. application-eval split, and the eight core LLM capabilities with their benchmarks*

## Why informal testing isn't enough

Most people who build an LLM-based application test it the same informal way: throw a handful of prompts at it, read the outputs, and decide it "feels" right. This is often called **vibe testing** — casually trying an application with a few prompts and judging it by feel rather than by any defined, repeatable process. It doesn't scale past a personal project, because there's no fixed dataset or criteria to rerun against the next version, and no way to tell whether a change actually made things better.

A few real incidents show what that gap costs once a system reaches production:

- **Air Canada's chatbot** told a customer he could book a full-fare ticket and claim a bereavement discount retroactively — a policy that didn't exist. A tribunal held the airline liable for the chatbot's answer. **Paper:** [2024 BCCRT 149 (Moffatt v. Air Canada)](https://www.canlii.org/en/bc/bccrt/doc/2024/2024bccrt149/2024bccrt149.html)
- **A Chevrolet dealership's chatbot** was talked into "agreeing" to sell a car for $1 after a user simply instructed it to accept anything the customer said, with no safeguards against the override.
- **A lawyer's brief in Mata v. Avianca** cited six court cases fabricated by ChatGPT, complete with invented names and quotes; the fabrication went undetected until the court tried to verify them. **Paper:** [Mata v. Avianca, Inc., 678 F. Supp. 3d 443 (S.D.N.Y. 2023)](https://en.wikipedia.org/wiki/Mata_v._Avianca,_Inc.)

None of these failures were exotic. They were caught late because nobody had a systematic, repeatable way of checking the system's outputs before real users relied on them — which is exactly what LLM evaluation is meant to fix.

## Defining LLM evals

LLM evals can be defined as **systematic, repeatable tests used to judge an LLM or an LLM-powered system against clear criteria.**

Three properties sit inside that definition, and each one matters on its own: the tests are systematic, they're repeatable, and they run against clearly defined criteria.

### Systematic

Being systematic means not vibe testing — not just typing in whatever five questions come to mind and deciding the answers look fine. Instead, it means building a proper dataset that deliberately covers the edge cases a system is likely to encounter.

For a chatbot in production, a practical way to build that dataset is to sample real conversations from actual users — for example, pulling 100 real user chats at random and turning them into a structured test set. Testing against that dataset produces a much closer picture of how the system actually behaves for real users than testing against a handful of questions someone made up on the spot.

### Repeatable

Being repeatable means that the same dataset and process can be applied identically no matter what changes underneath — a different prompt, a different model, a different retriever, a different chunking strategy. The evaluation setup itself doesn't change; only the system under test does.

This is what makes version comparison possible. If version 1 and version 2 of a system are both run against the same fixed test dataset, the difference in their scores tells you directly whether the system actually improved — a comparison that's meaningless without a fixed, repeatable process behind it.

### Clear criteria

The third property is that evaluation depends on explicitly defined criteria — what exactly is being judged, and by what standard.

For a chatbot answering questions grounded in a knowledge base, that criteria set might include: the answer must be factually correct, it must be explained simply, it must draw only from the retrieved knowledge base rather than the model's general knowledge, and it must be free of unsafe or offensive content. Without criteria like these stated up front, testing collapses back into vibe testing. With them, it becomes a proper evaluation.

## LLM evals are not just metrics

A common misconception — one worth clearing up early — is that "eval" means "metric." That association makes sense coming from a classical machine learning background, where evaluating a model usually meant reporting accuracy, precision, or recall. But an LLM eval is not a single number; it's the **entire testing setup** built to test a system.

That setup includes several things at once:

- **What's being tested** — for a RAG chatbot, that might specifically be the retriever component rather than the whole pipeline.
- **What criteria it's judged against** — for a retriever, that might be retrieval accuracy specifically.
- **The dataset** prepared for that evaluation.
- **When the test runs** — offline during development, or continuously after the system is live in production.
- **The tooling used to run it** — for example, a RAG-specific framework like Ragas when the system being tested is a RAG application.

Put together, all of these — the target, the criteria, the dataset, the timing, and the tooling — make up what "LLM evals" actually refers to. It's a complete testing setup, not a score.

## The questions LLM evals are meant to answer

The goal of an eval isn't to hand back a single score — it's to answer concrete, practical questions about a system, such as:

- Can this model be used for a particular task or application?
- Is this system good enough to ship to production?
- Did prompt version 2 actually improve on version 1?
- Is the RAG answer grounded in the retrieved context, or is it drifting from it?
- Is the agent completing its task correctly?
- Is the chatbot safe for real users?
- Is latency under control?

These are the kinds of questions a well-designed evaluation setup is built to answer — which is also why the setup has to be systematic and repeatable in the first place: a one-off vibe check can't answer any of them with confidence.

## Model evals vs. application evals

The definition above covers two different things at once: it refers to judging "an LLM" and judging "an LLM-powered system." That split points to a useful distinction — LLM evals can be divided into **model evals** and **application evals**.

- **Model evals** evaluate the model itself.
- **Application evals** evaluate the complete LLM-based application built on top of that model.

These two labels aren't standardized industry terminology — they're used here purely as a simplifying frame. In practice, the industry uses the umbrella term "LLM evals" for both, and which one is meant is usually clear from context. The underlying distinction, though, is worth keeping separate in your head: testing a model's raw capabilities is a different exercise from testing whether a specific application built on that model actually works.

```mermaid
flowchart TD
    A[LLM Evals] --> B[Model Evals]
    A --> C[Application Evals]

    B --> B1["Evaluate the<br/>model itself"]
    B1 --> B2["Benchmarks:<br/>MMLU, GSM8K,<br/>HumanEval, SWE-bench,<br/>IFEval, MMMU,<br/>Needle in a Haystack"]

    C --> C1["Evaluate the full<br/>system built on<br/>the model"]
    C1 --> C2["System level:<br/>response quality,<br/>latency, cost"]
    C1 --> C3["Component level:<br/>retriever, embedding<br/>model, reranker"]
```

## Model evals: testing raw capability

Model evals exist to test and evaluate the capabilities of a model, independent of any specific application. This is the exercise that happens every time a new LLM is released — providers benchmark the model, document how it performs, and publish results. It's why a new model's announcement is usually accompanied by a claim like "this model tops a particular leaderboard" or "achieves this percentage on a particular benchmark": that number comes directly from running the model through a fixed set of model evals and publishing the results.

## The capabilities models get tested on

Current-generation LLMs are typically evaluated across eight capability categories:

1. **Reasoning** — can the model work through a problem step by step?
2. **Knowledge** — does the model have accurate general and world knowledge up to its training cutoff?
3. **Basic math** — can the model solve mathematical problems?
4. **Coding** — can the model write functioning code?
5. **Instruction following** — if given a sequence of instructions, does the model follow all of them correctly?
6. **Long-context handling** — can the model pull the correct answer out of a very large context window?
7. **Multimodal understanding** — can the model understand and/or generate across images, text, and audio, not just text alone?
8. **Tool use** — can the model use external tools correctly?

Each of these has one or more benchmarks built specifically to measure it.

## Common benchmarks for each capability

A few of the benchmarks most often cited for these capabilities:

- **General knowledge and reasoning — MMLU.** Tests a model across a wide span of subjects — science, history, law, medicine — and scores its answers. **Paper:** [arxiv.org/abs/2009.03300](https://arxiv.org/abs/2009.03300)
- **Basic math — GSM8K.** A set of grade-school-level math word problems used to score a model's arithmetic and multi-step reasoning. **Paper:** [arxiv.org/abs/2110.14168](https://arxiv.org/abs/2110.14168)
- **Coding — HumanEval and SWE-bench.** HumanEval checks whether generated code passes hidden unit tests; SWE-bench goes further, testing whether a model can resolve real GitHub issues in real repositories. **Paper:** [arxiv.org/abs/2107.03374](https://arxiv.org/abs/2107.03374) (HumanEval), [arxiv.org/abs/2310.06770](https://arxiv.org/abs/2310.06770) (SWE-bench)
- **Instruction following — IFEval.** Scores a model on whether it correctly follows precise, objectively checkable instructions, such as word-count or keyword constraints. **Paper:** [arxiv.org/abs/2311.07911](https://arxiv.org/abs/2311.07911)
- **Long-context handling — Needle in a Haystack.** Places a short, specific fact somewhere inside a very long document and checks whether the model can retrieve it. **Reference:** [github.com/gkamradt/LLMTest_NeedleInAHaystack](https://github.com/gkamradt/LLMTest_NeedleInAHaystack)
- **Multimodal understanding — MMMU.** College-level questions spanning multiple disciplines that require interpreting images alongside text — charts, diagrams, tables, and similar visual material. **Paper:** [arxiv.org/abs/2311.16502](https://arxiv.org/abs/2311.16502)

Model evals like these are what sit behind every "state of the art" claim attached to a new model release — a fixed benchmark, run the same way every time, with a published score at the end.

## Why model evals matter even if you rarely run them

Benchmarking a newly released model, documenting its capabilities, and publishing leaderboard scores is work done by the frontier labs that build the model — not by the people building applications on top of it. Someone working as an AI engineer will rarely run a model eval directly.

Being able to read model evals is still valuable, though. Knowing what a benchmark measures and which models score well on it directly informs a practical decision every application has to make: which underlying LLM to build on. Literacy in model evals — understanding what the numbers mean, without ever needing to compute them — is what supports that choice.

## Application evals: why the model is only one component

Application evals exist because, in most real systems, the LLM is just one piece of a much larger whole. It's tempting to assume an LLM application is entirely defined by the model powering it — that the model is the "brain" and therefore the whole story. In practice, once an application grows beyond a toy example, a long list of other components determines whether it actually works:

- The **user interface**.
- The **system prompt**.
- **Tools and APIs** the system can call.
- The **orchestration logic** that routes control through the system — branching, parallel execution, and so on.
- **Guardrails** applied at the application level.
- **Output parsers**.
- **Memory and context management**.
- For a RAG system specifically: the **retrieval pipeline** — the embedding model, the vector database, and everything around them.
- Once deployed: the **monitoring setup** and the **feedback loop** that watches the system in production.

None of these are optional extras. They're why application evaluation is its own discipline, separate from evaluating the model in isolation.

## The smartphone analogy

Chip manufacturers like Qualcomm or MediaTek publish benchmark scores for each new processor to show how strong that chip is. But a strong processor doesn't guarantee a good smartphone. A phone also needs a solid camera system, operating system, sound system, graphics hardware, battery, and screen — all working well together. A great processor paired with a mediocre camera and a weak battery still adds up to a mediocre phone.

The same logic applies to LLM-based systems. Frontier labs hand over an already-evaluated model, the way a chip manufacturer hands over an already-benchmarked processor. Everything built around that model — and whether it holds up as a whole — is the responsibility of whoever builds the application on top of it.

## What application evals assess

Application evals assess the behavior and performance of an LLM-powered application, either at the level of the entire system or at the level of a specific component within it.

Take a RAG chatbot as an example. At the **system level**, the whole pipeline gets evaluated together — how good the final response is, what the latency looks like, what the cost per token comes out to. At the **component level**, each piece gets evaluated on its own — whether the retriever is finding the right documents, whether the embedding model is placing things sensibly in vector space, whether the reranker is ordering results correctly.

The core question application evals ask is different from the one model evals ask. Model evals ask, "can the model do this?" Application evals ask, "does the product actually work?" — a question about the whole system the model sits inside, not the model alone.

## The questions application evals are meant to answer

For a chatbot built on top of a specific knowledge base, application evals are what answer questions such as:

- Was the user's question actually answered correctly?
- Was the knowledge base used properly to construct that answer?
- Was the answer faithful to the source material, or did it drift from it?
- Was the explanation easy enough for a beginner to follow?
- Did the system hallucinate anything?
- Was the response fast enough?
- Is the chatbot safe for real users?

In everyday use, "LLM evaluation" almost always refers to this — application evaluation — rather than model evaluation. Model evals stay in the background as the work frontier labs publish; application evals are the ongoing, hands-on discipline of making sure a specific system, built around a specific model, actually does its job.